# 01 – Аудит базы данных Европы

Цель – установить состав, временное покрытие и ограничения исходной таблицы перед подготовкой собственной аналитической базы.

Источник: [Large aircraft landing dataset at 20 large europen airports](https://zenodo.org/records/10210178), Benoit Figuet, DOI `10.5281/zenodo.10210178`, версия v1 от 27.11.2023. Лицензия – CC-BY-4.0 по сохраненным метаданным Zenodo. ADS-B наблюдения получены авторами из OpenSky, обогащение – из EUROCONTROL Network Manager.

Одна строка содержит метаданные наблюдения полета с итоговой посадкой и может описывать несколько попыток захода. `GoA` означает наличие хотя бы одного отмеченного ухода, а не число уходов, аварийность или нарушение. Полные траектории и погодные наблюдения в файле отсутствуют.

Аудит проверяет весь исходный Parquet блоками по 100 000 строк. Никакие строки не исключаются, значения не исправляются. Отдельные диагностические выгрузки сохраняют исходные значения и номера строк. Графики и очистка выполняются следующими этапами.

In [1]:
# Проверяем окружение и находим корень независимо от папки запуска Jupyter
from pathlib import Path
import importlib
import json
import sys

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').is_file()
             and (p / '.git').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Запустите Jupyter из проекта или папки europe/notebooks')
REGION = ROOT / 'europe'
SOURCE = REGION / 'data/source/osn23_landings_merged.parquet.gzip'
REPORTS = REGION / 'reports'
sys.path.insert(0, str(REGION / 'scripts'))
import profile_data
profile_data = importlib.reload(profile_data)

pd.set_option('display.max_rows', 70)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 90)
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('pyarrow:', pa.__version__)
print('Источник:', SOURCE.relative_to(ROOT).as_posix())
print('Размер файла, МБ:', round(SOURCE.stat().st_size / 1_000_000, 1))

Python: 3.14.6
pandas: 3.0.6
pyarrow: 25.0.1
Источник: europe/data/source/osn23_landings_merged.parquet.gzip
Размер файла, МБ: 331.3


## Схема и первый просмотр

Сначала показываем фактические типы и десять исходных записей. Эти десять строк служат только примером структуры; все дальнейшие показатели рассчитываются по всему файлу.

[Русский словарь полей](../docs/data_dictionary.md) отделяет описание авторов от интерпретаций, которые еще требуют проверки. В схеме есть служебный индекс pandas; он не считается содержательным признаком или уникальным идентификатором.

In [2]:
# Читаем физическую схему, сохраняя все поля исходника
parquet = pq.ParquetFile(SOURCE)
schema = pd.DataFrame([{'Поле': f.name, 'Тип Arrow': str(f.type),
                       'Назначение': 'Служебный индекс' if f.name == '__index_level_0__' else 'Данные'}
                      for f in parquet.schema_arrow])
display(schema)
print('Физических столбцов:', len(parquet.schema_arrow))
print('Содержательных полей:', len(schema) - int((schema['Поле'] == '__index_level_0__').sum()))
print('Строк по метаданным Parquet:', parquet.metadata.num_rows)

,Поле,Тип Arrow,Назначение
0,flight_id,string,Данные
1,callsign,string,Данные
2,icao24,string,Данные
3,landing_time,"timestamp[ns, tz=UTC]",Данные
4,registration,string,Данные
5,typecode,string,Данные
6,icaoaircrafttype,string,Данные
7,airport,string,Данные
8,country,string,Данные
9,attempt_times,"list<element: timestamp[us, tz=UTC]>",Данные


Физических столбцов: 21
Содержательных полей: 20
Строк по метаданным Parquet: 7984254


In [3]:
# Выводим небольшой пример без преобразования списков в модельные признаки
sample = next(parquet.iter_batches(batch_size=10)).to_pandas(ignore_metadata=True)
display(sample[['flight_id', 'icao24', 'airport', 'landing_time', 'GoA',
                'n_attempts', 'attempt_times', 'ILS']])

,flight_id,icao24,airport,landing_time,GoA,n_attempts,attempt_times,ILS
0,EZY3092_21625,440031,EGSS,2019-01-01 00:15:21+00:00,False,1,[2019-01-01T00:15:21.000000],[22]
1,PGT77F_32108,4b8e46,LTFJ,2019-01-01 00:23:58+00:00,False,1,[2019-01-01T00:23:58.000000],[06L]
2,JTG260_22155,45ab49,EKCH,2019-01-01 00:28:15+00:00,False,1,[2019-01-01T00:28:15.000000],[22L]
3,TFL554_98026,484ad0,EHAM,2019-01-01 00:56:00+00:00,False,1,[2019-01-01T00:55:24.000000],[18R]
4,VKG799_43226,45d968,EKCH,2019-01-01 02:34:54+00:00,False,1,[2019-01-01T02:34:54.000000],[22L]
5,PGT9QZ_18985,4b8e0a,LTFJ,2019-01-01 02:35:46+00:00,False,1,[2019-01-01T02:35:46.000000],[06L]
6,PGT817_36817,4b9064,LTFJ,2019-01-01 02:46:20+00:00,False,1,[2019-01-01T02:46:20.000000],[06L]
7,PGT5NL_91689,4bb852,LTFJ,2019-01-01 02:53:14+00:00,False,1,[2019-01-01T02:53:14.000000],[06L]
8,PGT758_83517,4bb848,LTFJ,2019-01-01 02:57:13+00:00,False,1,[2019-01-01T02:57:13.000000],[06L]
9,PGT69B_92666,4bb853,LTFJ,2019-01-01 03:02:18+00:00,False,1,[2019-01-01T03:02:18.000000],[06L]


## Полная проверка исходника

Следующая ячейка запускает полный аудит и заново формирует отчеты в `europe/reports`. Скрипт проверяет SHA256 по реестру источников до чтения и повторно после завершения. Для воспроизведения всех таблиц нужно выполнить ячейки по порядку.

[Код полного аудита](../scripts/profile_data.py) включает проверки вложенных списков, хронологии событий и точное сравнение кандидатов повторов. `data/review` содержит диагностические копии строк, а не очищенную базу.

In [4]:
# Выполняем полный проход; исходный Parquet открывается только для чтения
summary = profile_data.profile_source()
assert summary['source_unchanged']
assert summary['rows_deleted'] == 0 and summary['values_corrected'] == 0

def report(name):
    return pd.read_csv(REPORTS / name, keep_default_na=False)

LABELS = {'rows':'Кол-во записей', 'go_arounds':'С уходом', 'ga_percent':'Доля с уходом, %',
          'country':'Страна', 'airport':'Аэропорт', 'city':'Город', 'month':'Месяц', 'year':'Год',
          'n_attempts':'Кол-во попыток', 'reason':'Диагностическая причина', 'percent':'Доля записей, %'}

def show(frame):
    display(frame.rename(columns=LABELS).style.format(
        {LABELS.get(c,c):'{:.1f}' for c in ['ga_percent','percent'] if c in frame.columns}))

overview = pd.DataFrame([
    ['Кол-во записей', f"{summary['rows']:,}"],
    ['Кол-во содержательных полей', summary['business_columns']],
    ['Аэропортов', summary['airports']],
    ['Стран', summary['countries']],
    ['Записей с уходом', f"{summary['go_arounds']:,}"],
    ['Доля с уходом, %', f"{summary['ga_percent']:.4f}"],
    ['Начало периода UTC', summary['min_landing_time']],
    ['Конец периода UTC', summary['max_landing_time']],
], columns=['Показатель', 'Значение'])
display(overview)

Проверено 1,000,000 строк.


Проверено 2,000,000 строк.


Проверено 3,000,000 строк.


Проверено 4,000,000 строк.


Проверено 5,000,000 строк.


Проверено 6,000,000 строк.


Проверено 7,000,000 строк.


{
  "audit_date": "2026-10-08",
  "source": "europe/data/source/osn23_landings_merged.parquet.gzip",
  "source_bytes": 331328945,
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "source_unchanged": true,
  "rows": 7984254,
  "physical_columns": 21,
  "business_columns": 20,
  "airports": 20,
  "countries": 13,
  "go_arounds": 26253,
  "ga_percent": 0.328809679651975,
  "min_landing_time": "2019-01-01 00:15:21+00:00",
  "max_landing_time": "2023-05-30 23:58:59+00:00",
  "unique_flagged_rows": 7540,
  "flagged_go_arounds": 1120,
  "rows_deleted": 0,
  "values_corrected": 0,
  "repeated_flight_ids": 23889,
  "excess_flight_id_rows": 29238,
  "repeated_core_keys": 0,
  "excess_core_key_rows": 0,
  "full_duplicate_rows_excluding_storage_index": 0,
  "max_n_attempts": 33.0,
  "rules_status": "diagnostic_not_cleaning_rules",
  "runtime": {
    "python": "3.14.6",
    "pandas": "3.0.6",
    "pyarrow": "25.0.1"
  }
}


,Показатель,Значение
0,Кол-во записей,"7,984,254"
1,Кол-во содержательных полей,20
2,Аэропортов,20
3,Стран,13
4,Записей с уходом,"26,253"
5,"Доля с уходом, %",0.3288
6,Начало периода UTC,2019-01-01 00:15:21+00:00
7,Конец периода UTC,2023-05-30 23:58:59+00:00


## Географический состав

Показываем все страны и все аэропорты исходного файла. Города добавлены как подписи для чтения, а не как исходные признаки. Европейский охват здесь соответствует выборке авторов; принадлежность к этой выборке не доказывает единообразие процедур и условий наблюдения.

In [5]:
# Состав по странам; суммы должны совпасть с полным набором
countries = report('country_summary.csv').sort_values('rows', ascending=False)
assert int(countries['rows'].sum()) == summary['rows']
assert int(countries['go_arounds'].sum()) == summary['go_arounds']
show(countries)

,Страна,Кол-во записей,С уходом,"Доля с уходом, %"
3,Germany,1360675,4492,0.3
12,United Kingdom,1189668,4273,0.4
2,France,1004272,3621,0.4
5,Italy,687865,1392,0.2
6,Netherlands,639766,1414,0.2
10,Switzerland,627567,2246,0.4
8,Spain,592140,1257,0.2
11,Turkey,387368,2414,0.6
0,Austria,360091,1375,0.4
4,Ireland,313008,1352,0.4


In [6]:
# Все аэропорты с городами и частотами отмеченных событий
airports = report('airport_summary.csv').sort_values('rows', ascending=False)
assert int(airports['rows'].sum()) == summary['rows']
assert int(airports['go_arounds'].sum()) == summary['go_arounds']
show(airports)
print('Аэропортов с несколькими значениями страны:', int((airports.groupby('airport')['country'].nunique() > 1).sum()))

,Страна,Аэропорт,Город,Кол-во записей,С уходом,"Доля с уходом, %"
2,France,LFPG,Париж,701670,2653,0.4
18,United Kingdom,EGLL,Лондон,686819,2607,0.4
5,Germany,EDDF,Франкфурт-на-Майне,644881,2161,0.3
10,Netherlands,EHAM,Амстердам,639766,1414,0.2
12,Spain,LEMD,Мадрид,592140,1257,0.2
6,Germany,EDDM,Мюнхен,499206,1532,0.3
9,Italy,LIRF,Рим,391269,672,0.2
16,Turkey,LTFJ,Стамбул,387368,2414,0.6
15,Switzerland,LSZH,Цюрих,372304,1486,0.4
0,Austria,LOWW,Вена,360091,1375,0.4


Аэропортов с несколькими значениями страны: 0


Частота отмеченных уходов условна относительно состава и покрытия этого источника. Сравнение аэропортов по одной доле не является оценкой их безопасности. Отбор по критерию точности и исключения из прежнего эксперимента здесь не применяются.

## Временное покрытие

Группируем записи по времени итоговой посадки в UTC. Частичный 2023 год нельзя сравнивать с полными годами без учета сезонности. Наличие наблюдений в месяце не означает полноту всех его дней и полетов.

Описание Zenodo указывает период до июня 2023 года. Ниже проверяется фактический период локального файла; при расхождении сохраняем оба утверждения и не дополняем данные догадками.

In [7]:
# Показываем каждый представленный год
annual = report('year_summary.csv')
show(annual)
assert int(annual['rows'].sum()) == summary['rows']

,Год,Кол-во записей,С уходом,"Доля с уходом, %"
0,2019,2516384,7235,0.3
1,2020,1109554,4105,0.4
2,2021,1368238,4637,0.3
3,2022,2197853,7466,0.3
4,2023,792225,2810,0.4


In [8]:
# Показываем все месяцы фактического периода
monthly = report('month_summary.csv')
show(monthly)
assert int(monthly['rows'].sum()) == summary['rows']
print('Фактический последний месяц:', monthly['month'].max())

,Месяц,Кол-во записей,С уходом,"Доля с уходом, %"
0,2019-01,174478,558,0.3
1,2019-02,180291,501,0.3
2,2019-03,208716,762,0.4
3,2019-04,207116,576,0.3
4,2019-05,219840,584,0.3
5,2019-06,218949,656,0.3
6,2019-07,231785,653,0.3
7,2019-08,232654,597,0.3
8,2019-09,218240,591,0.3
9,2019-10,232337,649,0.3


Фактический последний месяц: 2023-05


In [9]:
# Проверяем дни наблюдений отдельно для каждого аэропорта
coverage = report('coverage_summary.csv')
display(coverage.rename(columns={'airport':'Аэропорт', 'observed_days':'Дней с записями',
    'calendar_days':'Дней в общем периоде', 'days_without_records':'Дней без записей',
    'observed_day_percent':'Доля дней с записями, %', 'first_calendar_day':'Начало общего периода',
    'last_calendar_day':'Конец общего периода'}).style.format({'Доля дней с записями, %':'{:.1f}'}))
print('Подробные таблицы: airport_month_summary.csv и airport_calendar.csv.')

,Аэропорт,Дней с записями,Дней в общем периоде,Начало общего периода,Конец общего периода,Дней без записей,"Доля дней с записями, %"
0,EDDB,1587,1611,2019-01-01,2023-05-30,24,98.5
1,EDDF,1464,1611,2019-01-01,2023-05-30,147,90.9
2,EDDM,1585,1611,2019-01-01,2023-05-30,26,98.4
3,EGCC,1467,1611,2019-01-01,2023-05-30,144,91.1
4,EGLL,1584,1611,2019-01-01,2023-05-30,27,98.3
5,EGSS,1587,1611,2019-01-01,2023-05-30,24,98.5
6,EHAM,1465,1611,2019-01-01,2023-05-30,146,90.9
7,EIDW,1586,1611,2019-01-01,2023-05-30,25,98.4
8,EKCH,1465,1611,2019-01-01,2023-05-30,146,90.9
9,EPWA,1588,1611,2019-01-01,2023-05-30,23,98.6


Подробные таблицы: airport_month_summary.csv и airport_calendar.csv.


День без записей означает отсутствие наблюдений в файле. Причиной могут быть отбор авторов, доступность ADS-B, изменение работы аэропорта или другие обстоятельства. По этой таблице нельзя установить причину или фактическую интенсивность движения.

## Поля и представления пропусков

Раздельно считаем физический null, пустую строку, строку из пробелов и текстовые маркеры отсутствия. Для списков дополнительно проверяем пустые списки и null внутри. Не заменяем эти значения нулями и не объединяем их автоматически.

In [10]:
# Пропуски по каждому физическому полю
missing = report('missingness.csv')
display(missing.rename(columns={'column':'Поле', 'arrow_type':'Тип Arrow', 'null_count':'Null',
    'null_percent':'Null, %', 'empty_string_count':'Пустая строка',
    'whitespace_only_count':'Только пробелы', 'text_missing_marker_count':'Текстовый маркер',
    'unique_nonnull_strings':'Уникальных строк без null', 'role':'Роль'}).style.format({'Null, %':'{:.1f}'}))

,Поле,Тип Arrow,Null,"Null, %",Пустая строка,Только пробелы,Текстовый маркер,Уникальных строк без null,Роль
0,flight_id,string,0,0.0,0,0,0,,data
1,callsign,string,0,0.0,0,0,0,186066.0,data
2,icao24,string,0,0.0,0,0,0,24054.0,data
3,landing_time,"timestamp[ns, tz=UTC]",0,0.0,0,0,0,,data
4,registration,string,859077,10.8,242227,0,0,17937.0,data
5,typecode,string,859077,10.8,475015,0,0,386.0,data
6,icaoaircrafttype,string,859077,10.8,543159,0,0,39.0,data
7,airport,string,0,0.0,0,0,0,20.0,data
8,country,string,0,0.0,0,0,0,13.0,data
9,attempt_times,list,0,0.0,0,0,0,,data


In [11]:
# Диапазоны числовых полей, без исправления значений
numeric = report('numeric_ranges.csv')
display(numeric.rename(columns={'column':'Поле', 'count':'Конечных значений',
    'nonfinite':'Неконечных непустых значений', 'min':'Минимум', 'max':'Максимум'}))

,Поле,Конечных значений,Неконечных непустых значений,Минимум,Максимум
0,n_attempts,7984254,0,1.000000,33.000000
1,C40_BEARING,6331872,0,0.000000,359.988163
2,C40_CROSS_LAT,6331872,0,39.805278,60.319722
3,C40_CROSS_LON,6331872,0,-7.393611,22.058889


Диапазоны широты, долготы и направления используются для диагностики. Они не подтверждают правильность конкретной точки входа. Метеоданные, высота и скорость самолета в схеме отсутствуют; проверки температуры и давления из старого эксперимента сюда не переносятся.

## Категории воздушных судов и сегменты перевозок

В `icaoaircrafttype` обнаружены значения разного формата: например, `L2J` рядом с `A332` и `a320`. В диагностике выделяются непустые значения вне преобладающего шаблона «буква + цифра + буква» и совпадения с `typecode` без учета регистра. Это проверка структуры поля, а не нормативное правило и не автоматическое исправление.

Физический null в таблицах обозначен как `<NULL>` только для отображения. Пустая строка сохраняется как отдельная категория. Статус справочников, историческая актуальность и доступность этих сведений в момент будущего прогноза требуют проверки.

In [12]:
# Все представленные сегменты перевозок
show(report('market_segment_summary.csv'))

,market_segment,Кол-во записей,С уходом,"Доля с уходом, %"
0,Mainline,3239614,10524,0.3
1,Lowcost,1759318,5694,0.3
2,,1515276,5688,0.4
3,Regional,846628,1986,0.2
4,Business,250875,826,0.3
5,Cargo,229537,772,0.3
6,Non-Scheduled,104147,259,0.2
7,Other,31507,437,1.4
8,Military,7269,67,0.9
9,Scheduled,83,0,0.0


In [13]:
# Классы воздушных судов и составные классификационные коды
show(report('AC_CLASS_summary.csv'))
show(report('icaoaircrafttype_summary.csv'))

,AC_CLASS,Кол-во записей,С уходом,"Доля с уходом, %"
0,MJ,4995279,14461,0.3
1,,1652382,7847,0.5
2,H,1169202,3560,0.3
3,MT,108719,239,0.2
4,LJ,45305,105,0.2
5,LT,11997,36,0.3
6,ZZ,702,0,0.0
7,LP,645,5,0.8
8,HEL,23,0,0.0


,icaoaircrafttype,Кол-во записей,С уходом,"Доля с уходом, %"
0,L2J,6258343,19220,0.3
1,,859077,3258,0.4
2,,543159,1837,0.3
3,L4J,161100,667,0.4
4,L2T,122898,680,0.6
5,L3J,16911,107,0.6
6,L1T,9235,28,0.3
7,L1P,3613,21,0.6
8,H2T,2503,330,13.2
9,A332,1140,1,0.1


In [14]:
# Типы самолетов: первые 20 по объему, полный перечень сохранен в отчете
types = report('typecode_summary.csv')
show(types.head(20))
print('Категорий typecode, включая представления пропусков:', len(types))

,typecode,Кол-во записей,С уходом,"Доля с уходом, %"
0,A320,1350700,4274,0.3
1,B738,1170953,4050,0.3
2,,859077,3258,0.4
3,A319,618064,1668,0.3
4,,475015,1555,0.3
5,A321,429034,1584,0.4
6,A20N,338285,1096,0.3
7,E190,187962,481,0.3
8,E195,185683,444,0.2
9,B77W,176380,667,0.4


Категорий typecode, включая представления пропусков: 387


## Разметка и попытки захода

Проверяем распределение `n_attempts`, длины `attempt_times` и `ILS`, а также согласованность с `GoA`. Совпадение метки с условием `n_attempts > 1` показывает внутреннюю согласованность полей, но не является независимой проверкой событий.

Больше двух попыток – отдельная диагностическая группа. Она не объявляется учебными полетами и не исключается из базы.

In [15]:
# Полное распределение количества попыток
attempts = report('attempts_summary.csv')
show(attempts)
assert int(attempts['rows'].sum()) == summary['rows']

,Кол-во попыток,Кол-во записей,С уходом,"Доля с уходом, %"
0,1,7958001,0,0.0
1,2,25201,25201,100.0
2,3,421,421,100.0
3,4,104,104,100.0
4,5,81,81,100.0
5,6,69,69,100.0
6,7,38,38,100.0
7,8,21,21,100.0
8,9,24,24,100.0
9,10,19,19,100.0


In [16]:
# Длины списков попыток и обозначений направлений
lengths = report('list_lengths.csv')
display(lengths.rename(columns={'field':'Поле', 'list_length':'Длина списка', 'rows':'Кол-во записей'}))

,Поле,Длина списка,Кол-во записей
0,attempt_times,1,7958001
1,attempt_times,2,25201
2,attempt_times,3,421
3,attempt_times,4,104
4,attempt_times,5,81
5,attempt_times,6,69
6,attempt_times,7,38
7,attempt_times,8,21
8,attempt_times,9,24
9,attempt_times,10,19


## Повторы идентификаторов и записей

Проверяем `flight_id` отдельно от составного ключа `landing_time + icao24 + airport`. Хеши используются только для отбора кандидатов. Итоговые совпадения устанавливаются по точным значениям ключа и всех содержательных полей, включая списки. Служебный индекс pandas исключен из определения содержательного дубля.

Совпадение идентификатора, ключа или даже всех полей требует отдельного решения об обработке; этот аудит не удаляет записи.

In [17]:
# Выводим точные результаты проверки повторов
with (REPORTS / 'duplicate_check.json').open(encoding='utf8') as stream:
    duplicates = json.load(stream)
display(pd.DataFrame([
    ['Повторяющихся flight_id', duplicates['flight_id_repeated_values']],
    ['Дополнительных строк с повтором flight_id', duplicates['flight_id_excess_rows']],
    ['Повторяющихся составных ключей', duplicates['repeated_core_keys']],
    ['Строк в группах повторяющегося составного ключа', duplicates['rows_in_repeated_core_keys']],
    ['Дополнительных строк с повтором составного ключа', duplicates['excess_core_key_rows']],
    ['Дополнительных полных дублей без служебного индекса', duplicates['full_duplicate_rows_excluding_storage_index']],
], columns=['Показатель', 'Кол-во']))
if duplicates['exact_core_key_examples']:
    display(pd.DataFrame(duplicates['exact_core_key_examples'])[
        ['source_row','flight_id','icao24','airport','landing_time','n_attempts','GoA']])

,Показатель,Кол-во
0,Повторяющихся flight_id,23889
1,Дополнительных строк с повтором flight_id,29238
2,Повторяющихся составных ключей,0
3,Строк в группах повторяющегося составного ключа,0
4,Дополнительных строк с повтором составного ключа,0
5,Дополнительных полных дублей без служебного индекса,0


In [18]:
# Примеры повторяющихся flight_id; совпадение имени не означает один и тот же полет
identifier_examples = pd.DataFrame(duplicates['repeated_identifier_examples'])
if not identifier_examples.empty:
    display(identifier_examples[['flight_id','icao24','airport','landing_time']].head(15))

,flight_id,icao24,airport,landing_time
0,MSR791_079,0100de,LIRF,2019-01-01 11:38:28+00:00
1,MSR797_104,01010c,LOWW,2019-01-01 12:14:38+00:00
2,NIA191_104,0101b4,LTFJ,2019-01-02 13:37:11+00:00
3,NIA191_105,0101b4,LTFJ,2019-01-04 12:24:54+00:00
4,MSR757_023,0100a8,EHAM,2019-01-05 13:22:53+00:00
5,MSR771_036,0100df,LSGG,2019-01-06 12:35:28+00:00
6,MSR731_099,010108,EDDB,2019-01-06 13:06:05+00:00
7,MSR731_054,0100de,EDDB,2019-01-08 12:53:40+00:00
8,NIA191_107,0101b4,LTFJ,2019-01-11 11:51:18+00:00
9,MSR771_036,01010d,LSGG,2020-01-14 12:10:20+00:00


## Время входа в ASMA и пригодность для прогнозирования

`C40_CROSS_TIME` заявлено как время входа в область радиусом 40 NM. Проверяем наличие координат, положение этого времени относительно первой попытки и итоговой посадки. Разность времени посадки и входа приводится только как диагностический показатель.

Отсутствие флага не доказывает доступность сведений в реальном времени. До обучения потребуется определить момент прогноза, источник признаков и правила работы с пропусками.

In [19]:
# Сводка времени между входом в ASMA и итоговой посадкой
with (REPORTS / 'c40_to_landing_duration.json').open(encoding='utf8') as stream:
    c40_duration = json.load(stream)
display(pd.DataFrame([
    ['Записей с обоими временами', c40_duration['rows_with_both_times']],
    ['Минимум, мин', c40_duration['min']],
    ['1-й процентиль, мин', c40_duration['quantiles'].get('0.01')],
    ['Медиана, мин', c40_duration['quantiles'].get('0.5')],
    ['99-й процентиль, мин', c40_duration['quantiles'].get('0.99')],
    ['Максимум, мин', c40_duration['max']],
], columns=['Показатель','Значение']))

,Показатель,Значение
0,Записей с обоими временами,6.331872e+06
1,"Минимум, мин",-9.200000e+00
2,"1-й процентиль, мин",9.633333e+00
3,"Медиана, мин",1.458333e+01
4,"99-й процентиль, мин",2.943333e+01
5,"Максимум, мин",1.647667e+02


## Диагностические флаги и примеры

Проверки охватывают обязательные значения, формат адресов и аэропортов, согласованность количества попыток и списков, порядок времен, время входа в ASMA и диапазоны координат. Формат обозначения ВПП проверяется отдельно от исторической применимости процедуры.

Флаг – основание для разбора. Он не доказывает ошибку, нарушение, учебное назначение полета или опасность захода. Причины могут пересекаться, поэтому суммы строк по флагам не равны числу уникальных отмеченных записей.

In [20]:
# Только ненулевые группы, полный перечень сохранен в CSV
flags = report('review_flag_summary.csv')
show(flags.loc[flags['rows'] > 0])
print('Уникальных записей с хотя бы одним флагом:', f"{summary['unique_flagged_rows']:,}")
print('Среди них с уходом:', f"{summary['flagged_go_arounds']:,}")

,Диагностическая причина,Кол-во записей,С уходом,"Доля записей, %"
0,icaoaircrafttype:outside_dominant_description_pattern,6407,29,0.1
1,n_attempts:more_than_two,1052,1052,0.0
2,icaoaircrafttype:equals_typecode_ignoring_case,130,0,0.0
3,C40_CROSS_TIME:after_first_attempt,87,45,0.0
4,C40_CROSS_TIME:after_landing,42,0,0.0


Уникальных записей с хотя бы одним флагом: 7,540
Среди них с уходом: 1,120


In [21]:
# По одному примеру на причину; все исходные поля сохранены в диагностическом Parquet
with (REPORTS / 'review_examples.json').open(encoding='utf8') as stream:
    examples = pd.DataFrame(json.load(stream))
if not examples.empty:
    display(examples.drop_duplicates('review_reason')[
        ['review_reason','source_row','flight_id','airport','landing_time','n_attempts','GoA']])
print('Номер source_row начинается с нуля и соответствует порядку строк Parquet.')

,review_reason,source_row,flight_id,airport,landing_time,n_attempts,GoA
0,n_attempts:more_than_two,11474,FCK311_163754,EDDF,2019-01-03 00:57:06+00:00,4,True
3,icaoaircrafttype:outside_dominant_description_pattern,661,VLG442_25252,LIRF,2019-01-01 07:49:29+00:00,1,False
6,icaoaircrafttype:equals_typecode_ignoring_case,478536,N51VE_123804,LSZH,2019-03-19 11:02:24+00:00,1,False
8,C40_CROSS_TIME:after_landing,696773,VLG6273_23112,LIRF,2019-04-19 22:05:09+00:00,1,False
9,C40_CROSS_TIME:after_first_attempt,696773,VLG6273_23112,LIRF,2019-04-19 22:05:09+00:00,1,False


Номер source_row начинается с нуля и соответствует порядку строк Parquet.


## Сохранность и вывод аудита

Проверка исходника выполняется независимо от диагностических отчетов. Очищенный файл не создается. Сохраненные результаты относятся к исходному набору; будущие правила обработки должны иметь отдельное обоснование и версию.

In [22]:
# Подтверждаем сохранность исходного файла и согласованность итогов
assert profile_data.sha256(SOURCE) == summary['source_sha256']
assert summary['source_unchanged']
assert summary['rows_deleted'] == 0
assert summary['values_corrected'] == 0
for name in ['country_summary.csv','airport_summary.csv','month_summary.csv','year_summary.csv','attempts_summary.csv']:
    table = report(name)
    assert int(table['rows'].sum()) == summary['rows'], name
    assert int(table['go_arounds'].sum()) == summary['go_arounds'], name
print('Исходный Parquet сохранен без изменений. Суммы агрегатов сходятся.')

display(Markdown(
    f"Проверены **{summary['rows']:,} записей**, **{summary['airports']} аэропортов**, "
    f"**{summary['countries']} стран**. Записей с отмеченным уходом: **{summary['go_arounds']:,}** "
    f"(**{summary['ga_percent']:.4f}%**). Фактический период UTC: "
    f"**{summary['min_landing_time']} – {summary['max_landing_time']}**. "
    f"Отдельного разбора требуют **{summary['unique_flagged_rows']:,} записей с диагностическими флагами**, "
    f"повторы идентификаторов и особенности временного покрытия. "
    f"Максимальное число попыток: **{int(summary['max_n_attempts'])}**. "
    f"Это результаты диагностики, а не решения об исключении."
))

Исходный Parquet сохранен без изменений. Суммы агрегатов сходятся.


Проверены **7,984,254 записей**, **20 аэропортов**, **13 стран**. Записей с отмеченным уходом: **26,253** (**0.3288%**). Фактический период UTC: **2019-01-01 00:15:21+00:00 – 2023-05-30 23:58:59+00:00**. Отдельного разбора требуют **7,540 записей с диагностическими флагами**, повторы идентификаторов и особенности временного покрытия. Максимальное число попыток: **33**. Это результаты диагностики, а не решения об исключении.

Следующий этап – разобрать пропуски, покрытие, группы повторов и флагов; проверить смысл спорных полей по источнику и согласовать обработку. После этого формируется производная аналитическая база.

Полные траектории и независимая проверка меток пока отсутствуют. Нельзя объявлять таблицу готовой для раннего прогнозирования или делать выводы о причинах ухода и безопасности аэропортов. Обучение моделей в этом ноутбуке не выполняется.